# 02 — Dataset Preparation

**Project:** Lightweight domain adaptation of small LLMs for chemistry using LoRA and QLoRA (MSc FYP).

**Phase goal (from supervisor's plan):**
1. Download BBBP, BACE, ESOL from MoleculeNet via DeepChem
2. Clean SMILES with RDKit; drop invalid molecules
3. Compute Bemis-Murcko scaffolds for every molecule
4. Implement scaffold-split (80 / 10 / 10)
5. Save splits as CSV (`bbbp_train.csv`, `bbbp_val.csv`, `bbbp_test.csv`, and same for BACE, ESOL)
6. Sanity check: class balance per split; no SMILES leak between train / test
7. (Commit done from outside the notebook.)

**Why scaffold split (worth knowing for the viva):** in random splits, train and test end up structurally similar, so the model can memorise rather than generalise. **Bemis-Murcko scaffolds** strip a molecule down to its core ring-system + linker (sidechains removed). Splitting by scaffold puts structurally distinct molecules in train vs test — a more honest test of generalisation, which is what drug-discovery cares about.

## 1. Install cheminformatics stack

On a fresh Colab VM, `rdkit` isn't installed. We use RDKit for SMILES parsing, canonicalisation, and Bemis-Murcko scaffolds. `pandas` and `scikit-learn` come pre-installed but we list them for clarity.

(We don't need `deepchem` — MoleculeNet's CSVs are publicly hosted, so we pull them straight with pandas.)

In [ ]:
%pip install -q rdkit pandas scikit-learn

In [ ]:
import os, sys, random
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict

RDLogger.DisableLog("rdApp.*")  # silence noisy RDKit parsing warnings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("pandas :", pd.__version__)

## 2. Load BBBP, BACE, ESOL from MoleculeNet

MoleculeNet hosts its CSVs on a public S3 bucket. We download each one directly with pandas — no DeepChem, no featurizer quirks, just `(SMILES, label)` rows.

**The three datasets:**
- **BBBP** (Blood-Brain Barrier Penetration) — binary classification, ~2,039 molecules. Label `p_np` = 1 if the molecule penetrates the blood-brain barrier.
- **BACE** — binary classification, ~1,513 molecules. Label `Class` = 1 if the molecule inhibits the BACE-1 enzyme (Alzheimer's drug target).
- **ESOL** — regression, ~1,128 molecules. Target = aqueous solubility (log mol/L).

In [ ]:
MOLNET_BASE = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets"

DATASETS = {
    "BBBP": {"url": f"{MOLNET_BASE}/BBBP.csv",  "smiles_col": "smiles", "label_col": "p_np"},
    "BACE": {"url": f"{MOLNET_BASE}/bace.csv",  "smiles_col": "mol",    "label_col": "Class"},
    "ESOL": {"url": f"{MOLNET_BASE}/delaney-processed.csv",
             "smiles_col": "smiles",
             "label_col": "measured log solubility in mols per litre"},
}

def load_raw(name):
    """Download a MoleculeNet CSV directly and return a (smiles, label) DataFrame."""
    spec = DATASETS[name]
    raw  = pd.read_csv(spec["url"])
    df   = pd.DataFrame({
        "smiles":  raw[spec["smiles_col"]].astype(str),
        "label":   raw[spec["label_col"]],
        "dataset": name,
    })
    print(f"{name:6s}: {len(df):5d} rows")
    return df

df_bbbp = load_raw("BBBP")
df_bace = load_raw("BACE")
df_esol = load_raw("ESOL")

In [ ]:
for df, name in [(df_bbbp, "BBBP"), (df_bace, "BACE"), (df_esol, "ESOL")]:
    print(f"\n--- {name} ---")
    print(df.head(3).to_string(index=False))

## 3. Clean SMILES with RDKit

Two kinds of bad rows we want to drop:
1. **Unparseable SMILES** — RDKit returns `None` (malformed strings, weird characters).
2. **Empty / duplicate** SMILES — confuse the splitter and bias evaluation.

We also **canonicalise** every SMILES. The same molecule can have multiple valid SMILES representations; canonicalisation gives one deterministic form so duplicates are detectable and downstream comparisons are fair.

In [ ]:
def canonicalise(smi):
    """Return canonical SMILES, or None if RDKit can't parse it."""
    if not isinstance(smi, str) or not smi.strip():
        return None
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

def clean(df, name):
    n0 = len(df)
    df = df.copy()
    df["smiles"] = df["smiles"].map(canonicalise)
    df = df.dropna(subset=["smiles"]).drop_duplicates(subset=["smiles"]).reset_index(drop=True)
    n1 = len(df)
    print(f"{name:6s}: {n0} → {n1} rows (dropped {n0 - n1} invalid / duplicate)")
    return df

df_bbbp = clean(df_bbbp, "BBBP")
df_bace = clean(df_bace, "BACE")
df_esol = clean(df_esol, "ESOL")

## 4. Compute Bemis-Murcko scaffolds

**What a scaffold is, concretely:** take a molecule, delete every atom that isn't part of a ring or a linker *between* rings. What remains is the Bemis-Murcko scaffold — the structural skeleton.

Examples (informal):
- Aspirin and ibuprofen share the **benzene** scaffold
- Caffeine and theobromine share the **xanthine** scaffold

Two molecules with the same scaffold are structurally similar; with different scaffolds they're structurally distinct. That's the key property we exploit in the split.

In [ ]:
def murcko_scaffold(smi):
    """Return Bemis-Murcko scaffold SMILES. Empty string = acyclic molecule (no rings)."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    scaf = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaf, canonical=True)

for df, name in [(df_bbbp, "BBBP"), (df_bace, "BACE"), (df_esol, "ESOL")]:
    df["scaffold"] = df["smiles"].map(murcko_scaffold)
    n_unique = df["scaffold"].nunique()
    print(f"{name:6s}: {len(df)} mols → {n_unique} unique scaffolds")

## 5. Scaffold-split (80 / 10 / 10)

**The algorithm (deterministic, no randomness):**
1. Group all molecules by their scaffold.
2. Sort scaffold-groups from largest to smallest.
3. Walk the sorted list: fill `train` until it has ~80% of molecules, then `val` until ~10%, rest to `test`.
4. Result: every scaffold lives in **exactly one** split. No molecule in test shares a scaffold with any molecule in train.

This is the canonical Bemis-Murcko scaffold split as used by MoleculeNet and most cheminformatics papers.

In [ ]:
def scaffold_split(df, frac_train=0.8, frac_val=0.1):
    """Bemis-Murcko scaffold split. Returns (train_df, val_df, test_df)."""
    groups = defaultdict(list)
    for idx, scaf in enumerate(df["scaffold"].tolist()):
        groups[scaf].append(idx)

    # Sort scaffold groups by size, largest first (standard convention).
    sorted_groups = sorted(groups.values(), key=lambda x: (len(x), x[0]), reverse=True)

    n_total = len(df)
    n_train_target = int(frac_train * n_total)
    n_val_target   = int(frac_val   * n_total)

    train_idx, val_idx, test_idx = [], [], []
    for group in sorted_groups:
        if len(train_idx) + len(group) <= n_train_target:
            train_idx.extend(group)
        elif len(val_idx) + len(group) <= n_val_target:
            val_idx.extend(group)
        else:
            test_idx.extend(group)

    return (
        df.iloc[train_idx].reset_index(drop=True),
        df.iloc[val_idx].reset_index(drop=True),
        df.iloc[test_idx].reset_index(drop=True),
    )

splits = {}
for df, name in [(df_bbbp, "BBBP"), (df_bace, "BACE"), (df_esol, "ESOL")]:
    tr, va, te = scaffold_split(df)
    splits[name] = (tr, va, te)
    print(f"{name:6s}: train={len(tr):4d}  val={len(va):4d}  test={len(te):4d}  "
          f"(ratios {len(tr)/len(df):.2f} / {len(va)/len(df):.2f} / {len(te)/len(df):.2f})")

## 6. Sanity checks

Two things to confirm before saving:
1. **Class balance per split** (BBBP & BACE are classification — make sure no split is all-positive or all-negative). For ESOL we print the target distribution instead.
2. **No SMILES leak** between train and test. Should be zero — if not, the split logic is broken.

In [ ]:
def class_balance(df, name):
    pos = int((df["label"] == 1).sum())
    neg = int((df["label"] == 0).sum())
    print(f"  {name:5s}: n={len(df):4d}  pos={pos:4d} ({pos/len(df):.2%})  neg={neg:4d} ({neg/len(df):.2%})")

for name in ["BBBP", "BACE"]:
    tr, va, te = splits[name]
    print(f"\n{name} class balance:")
    class_balance(tr, "train")
    class_balance(va, "val")
    class_balance(te, "test")

# ESOL is regression — print target distribution stats
tr, va, te = splits["ESOL"]
print("\nESOL target distribution (log mol/L):")
for split_name, split_df in [("train", tr), ("val", va), ("test", te)]:
    s = split_df["label"]
    print(f"  {split_name:5s}: n={len(split_df):4d}  mean={s.mean():.2f}  std={s.std():.2f}  "
          f"min={s.min():.2f}  max={s.max():.2f}")

In [ ]:
# Leak check: no SMILES should appear in both train and test.
print("SMILES leak check (train ∩ test should be empty):")
for name in ["BBBP", "BACE", "ESOL"]:
    tr, va, te = splits[name]
    leak_train_test = set(tr["smiles"]) & set(te["smiles"])
    leak_train_val  = set(tr["smiles"]) & set(va["smiles"])
    print(f"  {name:5s}: train∩test = {len(leak_train_test)}   train∩val = {len(leak_train_val)}")
    assert len(leak_train_test) == 0, f"{name} leaks between train and test!"
    assert len(leak_train_val)  == 0, f"{name} leaks between train and val!"
print("\nNo leaks. ✓")

## 7. Save splits as CSV

Saving to `/content/data/` (the Colab VM) plus mounting Google Drive so the CSVs persist across sessions. Later notebooks (`03_*`, `04_*`, ...) will read straight from Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT_DIR = "/content/drive/MyDrive/chemistry-peft-fyp/data"
os.makedirs(OUT_DIR, exist_ok=True)
print("Writing splits to:", OUT_DIR)

In [ ]:
for name in ["BBBP", "BACE", "ESOL"]:
    tr, va, te = splits[name]
    prefix = name.lower()
    tr.to_csv(f"{OUT_DIR}/{prefix}_train.csv", index=False)
    va.to_csv(f"{OUT_DIR}/{prefix}_val.csv",   index=False)
    te.to_csv(f"{OUT_DIR}/{prefix}_test.csv",  index=False)
    print(f"  {name}: saved {prefix}_train.csv ({len(tr)}), {prefix}_val.csv ({len(va)}), {prefix}_test.csv ({len(te)})")

print("\nFiles in output dir:")
for f in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(f"{OUT_DIR}/{f}") / 1024
    print(f"  {f}  ({size_kb:.1f} KB)")

## 8. Done

**Supervisor's checklist:**
- ✅ Downloaded BBBP, BACE, ESOL from MoleculeNet (direct CSV)
- ✅ Cleaned SMILES with RDKit; dropped invalid + duplicate molecules
- ✅ Computed Bemis-Murcko scaffolds
- ✅ Implemented scaffold-split (80 / 10 / 10)
- ✅ Saved splits as CSV per dataset
- ✅ Sanity-checked class balance + zero SMILES leak

**Next (Phase 03):** baseline evaluation — train a simple Random Forest on Morgan fingerprints for each dataset. That gives us the number to beat with LoRA/QLoRA in Phases 04/05.